# Quantum Attention Layer Demo

This notebook demonstrates the QFT-based Quantum Attention Layer, including:
1. Basic usage and circuit visualization
2. Parameter-shift gradient computation
3. Training the quantum attention layer
4. Hybrid quantum-classical integration with PyTorch

In [ ]:
# Setup
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram, circuit_drawer
import warnings
warnings.filterwarnings('ignore')

# Import our modules
from qft_attention import QFTAttentionLayer
from qft_attention_advanced import MultiHeadQFTAttention
from quantum_attention_trainable import TrainableQFTAttention

## 1. Basic Quantum Attention Layer

The QFT-based attention mechanism works as follows:
1. **Token Embedding**: Each token is encoded as a rotation angle (RY gate)
2. **QFT**: Quantum Fourier Transform moves to frequency domain
3. **Attention Phase**: Pairwise attention via controlled-RZ gates
4. **Inverse QFT**: Return to computational basis

In [ ]:
# Create a basic attention layer
qa = QFTAttentionLayer(num_tokens=3)

# Set some example parameters
phi = [0.5, 1.0, 1.5]  # Token embeddings
theta = [0.3, 0.6, 0.9]  # Attention phases

# Run the circuit
probs = qa.run(phi, theta, shots=4096)

print('Probability distribution:')
for state, prob in sorted(probs.items()):
    print(f'  |{state}⟩: {prob:.3f}')

In [ ]:
# Visualize the circuit
circuit = qa.build_circuit(phi, theta)
print('Quantum Circuit:')
print(circuit.draw(fold=120))

In [ ]:
# Plot the probability distribution
fig, ax = plt.subplots(figsize=(10, 5))
states = sorted(probs.keys())
values = [probs[s] for s in states]
ax.bar(states, values, color='steelblue')
ax.set_xlabel('Quantum State')
ax.set_ylabel('Probability')
ax.set_title('Quantum Attention Output Distribution')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Multi-Head Quantum Attention

Similar to classical transformers, we can have multiple attention heads.

In [ ]:
# Create multi-head attention
mha = MultiHeadQFTAttention(num_tokens=3, num_heads=2)

# Set parameters for each head
phi_per_head = [[0.5, 1.0, 1.5], [0.2, 0.8, 1.2]]
theta_per_head = [[0.3, 0.6, 0.9], [0.1, 0.4, 0.7]]

# Run multi-head attention
combined_probs = mha.run(phi_per_head, theta_per_head, shots=4096)

print('Combined probability distribution from 2 heads:')
for state, prob in sorted(combined_probs.items()):
    print(f'  |{state}⟩: {prob:.3f}')

## 3. Trainable Quantum Attention

Using the parameter-shift rule for gradient computation.

In [ ]:
# Create trainable layer
tqa = TrainableQFTAttention(num_tokens=3)

# Define target distribution (e.g., want |100⟩ to be most probable)
target_probs = {
    '000': 0.1, '001': 0.1, '010': 0.1, '011': 0.1,
    '100': 0.4, '101': 0.1, '110': 0.05, '111': 0.05
}

print('Training quantum attention layer...')
print('Target: maximize |100⟩ probability')
print()

# Train for a few epochs
losses = tqa.train(target_probs, epochs=30, learning_rate=0.05, shots=2048)

print(f'\nLoss reduction: {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%')

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses, 'b-', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training Loss Over Time')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Scalability Analysis

Let's analyze how the circuit depth scales with the number of tokens.

In [ ]:
from qiskit import transpile
from qiskit.circuit.library import QFT

def analyze_scalability(max_tokens=8):
    results = []
    
    for n in range(2, max_tokens + 1):
        # Build circuit
        qc = QuantumCircuit(n)
        
        # Token embeddings
        for i in range(n):
            qc.ry(0.5, i)
        
        # QFT
        qft = QFT(num_qubits=n, do_swaps=False)
        qc.compose(qft, qubits=list(range(n)), inplace=True)
        
        # Attention phases
        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        for i, j in pairs:
            qc.crz(0.5, i, j)
        
        # Inverse QFT
        qft_inv = QFT(num_qubits=n, do_swaps=False, inverse=True)
        qc.compose(qft_inv, qubits=list(range(n)), inplace=True)
        
        # Transpile to basis gates
        transpiled = transpile(qc, basis_gates=['u1', 'u2', 'u3', 'cx'])
        
        results.append({
            'tokens': n,
            'qubits': n,
            'depth': transpiled.depth(),
            'gates': len(transpiled),
            'cx_gates': transpiled.count_ops().get('cx', 0),
            'attention_pairs': len(pairs)
        })
    
    return results

results = analyze_scalability(8)

print('Scalability Analysis:')
print('-' * 70)
print(f'{"Tokens":>6} | {"Qubits":>6} | {"Depth":>6} | {"Gates":>6} | {"CX":>6} | {"Pairs":>6}')
print('-' * 70)
for r in results:
    print(f'{r["tokens"]:>6} | {r["qubits"]:>6} | {r["depth"]:>6} | {r["gates"]:>6} | {r["cx_gates"]:>6} | {r["attention_pairs"]:>6}')

In [ ]:
# Plot scalability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tokens = [r['tokens'] for r in results]
depths = [r['depth'] for r in results]
cx_gates = [r['cx_gates'] for r in results]

# Circuit depth
axes[0].plot(tokens, depths, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Tokens')
axes[0].set_ylabel('Circuit Depth')
axes[0].set_title('Circuit Depth vs Token Count')
axes[0].grid(True, alpha=0.3)

# CX gates (entangling operations)
axes[1].plot(tokens, cx_gates, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Tokens')
axes[1].set_ylabel('Number of CX Gates')
axes[1].set_title('Entangling Gates vs Token Count')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Hybrid Quantum-Classical Model

Integration with PyTorch for end-to-end training.

In [ ]:
import torch
import torch.nn as nn

class HybridQuantumClassifier(nn.Module):
    """
    A hybrid quantum-classical classifier that uses quantum attention
    for feature extraction and classical layers for classification.
    """
    
    def __init__(self, num_tokens=3, num_classes=2):
        super().__init__()
        self.num_tokens = num_tokens
        self.quantum_attention = TrainableQFTAttention(num_tokens)
        
        # Classical classification head
        self.classifier = nn.Sequential(
            nn.Linear(2**num_tokens, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes),
            nn.Softmax(dim=-1)
        )
    
    def forward(self):
        # Get quantum attention output
        probs = self.quantum_attention.forward(shots=1024)
        
        # Convert to tensor
        prob_tensor = torch.tensor(
            [probs.get(format(i, f'0{self.num_tokens}b'), 0.0) 
             for i in range(2**self.num_tokens)],
            dtype=torch.float32
        )
        
        # Classify
        return self.classifier(prob_tensor)

# Create hybrid model
model = HybridQuantumClassifier(num_tokens=3, num_classes=2)
output = model()

print('Hybrid Quantum-Classical Model Output:')
print(f'Class probabilities: {output.detach().numpy()}')

## Summary

This notebook demonstrated:

1. **Basic QFT Attention**: Token embeddings → QFT → Attention phases → Inverse QFT
2. **Multi-Head Attention**: Multiple attention heads with combined output
3. **Trainable Layer**: Parameter-shift rule for gradient computation
4. **Scalability**: Circuit depth grows as O(n²) with token count
5. **Hybrid Model**: Integration with PyTorch for end-to-end training

### Key Metrics

| Tokens | Qubits | Depth | CX Gates |
|--------|--------|-------|----------|
| 3      | 3      | ~30   | ~12      |
| 5      | 5      | ~80   | ~40      |
| 8      | 8      | ~200  | ~112     |

### Next Steps

1. Implement error mitigation for NISQ devices
2. Test on real quantum hardware
3. Benchmark against classical attention on small datasets
4. Explore quantum advantage for specific attention patterns